# 자연어 생성(NLG) 평가 지표 비교: METEOR vs BERTScore

본 교재는 기계 번역(MT) 및 자연어 생성(NLG) 모델의 성능을 평가하기 위해 사용되는 대표적인 두 가지 지표인 **METEOR**와 **BERTScore**의 수학적 원리, 작동 메커니즘, 그리고 한국어 자연어 처리 환경에서의 실제 적용 가능성을 심층적으로 다룹니다.

---

## 1. 서론: 자연어 생성 평가의 발전과 배경

자연어 생성 모델이 출력한 문장(Candidate)이 사람이 작성한 정답 문장(Reference)과 얼마나 유사한지 평가하는 것은 NLP 분야의 오랜 난제였습니다. 

초기 표준이었던 **BLEU(Bilingual Evaluation Understudy)** 는 단순한 N-gram 단어 일치도(정밀도)만을 측정하여 계산이 빠르다는 장점이 있었으나, 다음과 같은 치명적인 한계를 가졌습니다.
1. **재현율(Recall) 무시:** 모델이 정답 문장의 핵심 정보를 얼마나 많이 빠뜨렸는지 측정하지 못함.
2. **표면적 일치에 의존:** 의미가 완전히 같은 유의어나 형태소 변형(예: run - running)을 오답으로 처리함.
3. **어순 고려 부족:** 단어의 순서가 무작위로 바뀌어도 높은 점수가 나올 수 있음.

이러한 BLEU의 한계를 극복하기 위해 통계 및 규칙 기반으로 재현율과 유의어를 보완한 **METEOR(2005)** 가 등장하였고, 이후 딥러닝과 거대 언어 모델(PLM)의 시대를 맞아 문맥적 의미를 벡터 공간에서 비교하는 **BERTScore(2020)** 로 진화하게 되었습니다.

---

## 2. METEOR (Metric for Evaluation of Translation with Explicit ORdering)

### 2.1 개념 및 작동 원리
METEOR는 정답 문장과 생성 문장 간의 **정밀도(Precision)** 와 **재현율(Recall)** 을 모두 고려하되, 인간의 평가 경향을 반영하여 **재현율에 압도적인 가중치**를 부여하는 지표입니다. 또한 문장의 어순 차이를 계산하여 점수를 깎는 '페널티 시스템'을 도입했습니다.

### 2.2 수학적 수식 및 계산 과정

METEOR는 생성 문장과 정답 문장 간에 매칭된 단어 수(Unigram match)를 기반으로 다음과 같이 계산됩니다.

1. **정밀도(Precision, $P$) 및 재현율(Recall, $R$) 계산**
   $$
   P = \frac{m}{c}
   $$
   $$
   R = \frac{m}{r}
   $$
   * $m$: 매칭된 단어(Unigram)의 총 개수
   * $c$: 생성된 문장(Candidate)의 전체 단어 수
   * $r$: 정답 문장(Reference)의 전체 단어 수

2. **조화 평균 ($F_{mean}$) 계산**
   METEOR는 재현율($R$)에 정밀도($P$)보다 9배 더 큰 가중치를 부여하기 위해 아래와 같은 가중 조화 평균식을 사용합니다.
   $$
   F_{mean} = \frac{10PR}{R + 9P}
   $$

3. **페널티 ($Penalty$) 계산**
   단어들이 정답과 얼마나 일치하는 순서로 배열되었는지 측정합니다. 연속적으로 매칭된 단어들의 덩어리를 **청크(Chunk)** 라고 합니다. 문장이 잘게 쪼개질수록 청크의 개수($ch$)가 늘어나 페널티가 커집니다.
   $$
   Penalty = 0.5 \times \left(\frac{ch}{m}\right)^3
   $$
   * $ch$: 연속 매칭된 단어 구(Chunk)의 개수
   * $m$: 매칭된 단어의 총 개수

4. **최종 METEOR 점수 산출**
   $$
   METEOR = F_{mean} \times (1 - Penalty)
   $$

### 2.3 유의어 매칭(Synonym Match) 심층 분석
METEOR의 가장 큰 혁신은 정답 문장과 생성 문장의 단어를 선형적으로 짝짓는 **3단계 폭포수(Waterfall) 매칭 시스템**에 있습니다. 앞 단계에서 매칭되지 못한 단어들만 다음 단계로 넘어가 구제 기회를 얻습니다.

* **1단계: Exact Match (완전 일치)**
  * 스펠링과 형태가 완벽히 일치하는 단어를 매칭합니다.
  * *예: computer ↔ computer*
* **2단계: Stem Match (어간 일치)**
  * 형태소 분석(Porter Stemmer 등)을 통해 단어의 어미를 제거하고, 뿌리가 같은 단어를 동일어로 인정합니다.
  * *예: computes, computing, computed ↔ 모두 어간 'comput'로 일치 판정*
* **3단계: Synonym Match (유의어 매칭)**
  * 표면적인 형태와 어간이 모두 다르더라도, 외부 어휘 의미망인 **WordNet**을 조회하여 두 단어가 같은 **Synset(동의어 집합)**에 속해 있다면 100% 동일한 단어로 간주하고 매칭 성공($m$에 포함)으로 처리합니다.
  * *예: 정답 문장의 'fast'와 생성 문장의 'quick'은 스펠링이 전혀 다르지만, WordNet 상에서 [빠른, 신속한]의 동일 Synset에 속하므로 정답 인정.*

### 2.4 METEOR의 치명적 한계
1. **문맥 맹점 (Context-blindness):** 다의어를 처리할 때 문맥을 전혀 보지 않고 단어 단독으로 사전을 검색하므로 오매칭이 발생할 수 있습니다. (예: 금융 '은행' 문맥의 bank를 강둑의 'river'와 유의어로 엮는 오류)
2. **정적 사전의 한계:** WordNet 사전에 등록되지 않은 신조어, 도메인 전문 용어, 축약어는 매칭할 수 없습니다.
3. **자원 의존성:** 고도로 정제된 언어별 어휘 사전과 형태소 분석기가 없으면 다국어 확장이 불가능합니다.

---


## 3. BERTScore

### 3.1 개념 및 작동 원리
BERTScore는 사전 학습된 거대 언어 모델(Pre-trained Language Model, 예: BERT, RoBERTa)의 문맥적 임베딩(Contextual Embeddings)을 활용하는 현대적 평가 지표입니다. 문장을 단순히 분절된 단어의 집합으로 보지 않고, 주변 문맥이 모두 반영된 고차원 벡터 공간 내에서의 거리를 측정하여 의미적 유사도를 판별합니다.

### 3.2 수학적 수식 및 계산 과정

정답 문장 $x = \langle x_1, \dots, x_k \rangle$와 생성 문장 $\hat{x} = \langle \hat{x}_1, \dots, \hat{x}_l \rangle$이 주어졌을 때, 언어 모델을 거쳐 각 토큰은 임베딩 벡터 $\mathbf{x}_i$와 $\mathbf{\hat{x}}_j$로 변환됩니다. 두 벡터의 유사도는 **코사인 유사도(Cosine Similarity)** 를 이용해 계산하며, 이를 통해 정밀도와 재현율을 산출합니다.

1. **재현율 ($R_{BERT}$)**
   정답 문장의 각 토큰($x_i$)을 기준으로, 생성 문장($\hat{x}$)의 모든 토큰 벡터들 중 가장 코사인 유사도가 높은 값(Greedy Matching)을 찾아 평균을 냅니다.
   $$
   R_{BERT} = \frac{1}{|x|} \sum_{x_i \in x} \max_{\hat{x}_j \in \hat{x}} \mathbf{x}_i^\top \mathbf{\hat{x}}_j
   $$

2. **정밀도 ($P_{BERT}$)**
   생성 문장의 각 토큰($\hat{x}_j$)을 기준으로, 정답 문장($x$)의 모든 토큰 벡터들 중 가장 코사인 유사도가 높은 값을 찾아 평균을 냅니다.
   $$
   P_{BERT} = \frac{1}{|\hat{x}|} \sum_{\hat{x}_j \in \hat{x}} \max_{x_i \in x} \mathbf{x}_i^\top \mathbf{\hat{x}}_j
   $$

3. **최종 F1-Score ($F_{BERT}$)**
   정밀도와 재현율의 균형을 맞춘 조화 평균을 최종 점수로 사용합니다.
   $$
   F_{BERT} = 2 \frac{P_{BERT} \cdot R_{BERT}}{P_{BERT} + R_{BERT}}
   $$

*(참고: 실제 구현체에서는 가독성을 높이기 위해 데이터셋 전체의 최소 유사도 값을 기준으로 선형 변환하는 Importance Weighting 또는 Baseline Rescaling 기법을 적용하기도 합니다.)*

### 3.3 BERTScore의 주요 장점
1. **완벽한 문맥 이해:** "밥 먹었어?"와 "식사는 하셨습니까?"처럼 단어와 문장 구조가 전혀 달라도 문맥상 동의어임을 임베딩 공간에서 스스로 파악해 냅니다.
2. **사전 및 규칙 불필요:** 인위적인 유의어 사전 구축이나 형태소 규칙 정의가 필요하지 않으며, 언어 모델의 지식에 전적으로 의존합니다.
3. **높은 인간 상관관계:** 문장의 미묘한 뉘앙스 변화나 패러프레이징(Paraphrasing)을 정확히 캐치하여, 인간 평가단이 매긴 점수와 가장 유사한 경향성을 보입니다.

---

## 4. METEOR와 BERTScore의 종합 비교

| 비교 항목 | METEOR (2005) | BERTScore (2020) |
| :--- | :--- | :--- |
| **기반 기술** | N-gram 통계, 형태소 분석, 통계적 규칙 | 신경망 기반 문맥 임베딩 (Contextual Embedding) |
| **유의어 처리** | 외부 어휘 사전(WordNet) 매칭 | 벡터 공간 내 코사인 유사도 연산 (자체 파악) |
| **문맥(Context) 이해**| 불가능 (단어 단독 비교) | 매우 뛰어남 (주변 단어 관계 완전 고려) |
| **문장 구조 변형** | 청크 페널티 방식을 통해 제한적으로 반영 | 임베딩 벡터 내에 구조적 정보가 자연스럽게 반영됨 |
| **연산 비용 및 속도** | 매우 낮음 (CPU 환경에서 밀리초 단위 실행) | 높음 (딥러닝 모델 추론이 필요하여 GPU 권장) |
| **핵심 한계점** | 사전 구축의 한계, 다의어 오매칭 가능성 | 백본 언어 모델 성능에 종속됨, 절대 점수 왜곡 가능성 |

---






## 5. 한국어 자연어 처리(NLP) 환경에서의 적용 가능성 분석

### 5.1 METEOR의 한국어 적용 시 문제점과 한계
한국어는 어근에 조사와 어미가 결합하여 문법적 기능이 결정되는 **교착어**입니다. 이 특성 때문에 영어 중심의 METEOR를 그대로 쓰면 심각한 문제가 발생합니다.

1. **어절 단위 계산의 붕괴:** 문장을 단순히 공백(띄어쓰기) 기준으로 자르면 "학교에", "학교를", "학교가"는 모두 다른 단어로 인식되어 $P$와 $R$이 급격히 떨어집니다.
2. **형태소 분석기 전처리 필수:** 한국어에 사용하려면 반드시 KoNLPy(Mecab, Okt 등)를 도입하여 문장을 형태소 단위로 쪼갠 뒤 단어 매칭을 수행해야만 합니다.
3. **유의어 사전(WordNet) 연동 난제:** METEOR의 핵심인 3단계 '유의어 매칭'을 하려면 한국어 어휘 의미망인 **KorLex(한국어 워드넷)** 등이 필요합니다. 그러나 영어 WordNet만큼 오픈소스로 쉽게 통합되어 작동하는 라이브러리가 부족하여, 국내 많은 연구에서는 유의어 단계를 생략하고 1, 2단계(어간 일치) 수준에서만 타협하여 사용하는 한계가 있습니다.

### 5.2 BERTScore의 한국어 적용 시 장점 및 가이드라인
반면, BERTScore는 **한국어 자연어 생성 평가에 매우 강력하고 유연하게 적용**할 수 있어 학계와 실무에서 표준으로 자리 잡고 있습니다.

1. **규칙으로부터의 해방:** 형태소 분석기나 별도의 유의어 사전을 연동할 필요가 없습니다. 언어 모델의 토크나이저(예: WordPiece, Byte-Pair Encoding)가 단어를 서브워드 단위로 쪼개고, 모델이 문맥을 학습했기 때문에 교착어의 조사 변형이나 어미 변화를 유연하게 흡수합니다.
2. **한국어 특화 백본(Backbone) 모델 활용:**
   * 기본 다국어 모델인 `mBERT`나 `XLM-RoBERTa`를 지정해도 준수한 성능을 냅니다.
   * 성능을 극대화하기 위해 한국어 언어 특성을 깊게 학습한 **`KoBERT`**, **`KLUE-RoBERTa`**, 또는 **`KcBERT`**(구어체/댓글 특화) 등의 체크포인트를 BERTScore의 백본 모델로 직접 지정하여 사용할 수 있습니다. 이를 통해 한국어 특유의 존댓말 체계, 구어체 표현, 신조어까지 정확하게 평가할 수 있습니다.

### 5.3 실무자를 위한 최종 제언
* **속도와 비용이 최우선인 대규모 필터링 시스템:** 형태소 분석기를 연동한 간단한 형태의 METEOR(또는 BLEU)를 1차 스크리닝용으로 사용하는 것이 효율적일 수 있습니다.
* **고도화된 LLM 생성 능력 평가 (요약, 챗봇, 번역 등):** 단어 표면의 일치 여부보다 문맥과 의미 전달이 중요하므로, **한국어 특화 모델(예: KLUE-RoBERTa-Large)을 백본으로 탑재한 BERTScore를 평가지표로 채택하는 것이 정답(Human Evaluation)과 가장 높은 정합성**을 보장합니다.

---

## 6. 실습 예제 코드: 한국어 METEOR와 BERTScore 비교

아래 코드는 `PRJ02_W2_002_Generation_Metrics[Ans].ipynb`의 ROUGE/BLEU 예제와 같은 구조로, 하나의 정답 문장과 두 개의 생성 문장을 비교합니다. `generated_good`은 의미가 비슷하지만 표면 단어가 조금 다르고, `generated_bad`는 의미가 다른 문장입니다.

### 6.1 예시 데이터와 한국어 토크나이저 준비

In [1]:
from ranx_k.tokenizers import KiwiTokenizer

kiwi_tokenizer = KiwiTokenizer(use_stopwords=False, pos_filter=[])

reference = "오늘 날씨가 매우 좋습니다. 공원에서 산책하기 좋은 날이에요."
generated_good = "오늘은 날씨가 정말 좋네요. 공원에서 산책하면 좋을 것 같아요."
generated_bad = "비가 많이 오고 있어요. 실내에서 쉬는 게 좋을 것 같습니다."


def tokenize_ko(text: str) -> list[str]:
    return kiwi_tokenizer.tokenize(text)


print(tokenize_ko(reference))

['오늘', '날씨', '매우', '습니다', '공원', '에서', '산책', '에요']


### 6.2 한국어 토큰 기반 간이 METEOR 구현

`nltk.translate.meteor_score`는 WordNet 리소스가 요구됩니다. 아래 함수는 METEOR의 핵심 아이디어인 정밀도, 재현율, 청크 페널티를 한국어 토큰 기준으로 직접 계산합니다. 실제 연구용 METEOR와 달리 WordNet 기반 유의어 매칭은 포함하지 않습니다.

In [2]:
from collections import Counter


def count_token_matches(reference_tokens: list[str], candidate_tokens: list[str]) -> int:
    ref_counter = Counter(reference_tokens)
    cand_counter = Counter(candidate_tokens)
    return sum((ref_counter & cand_counter).values())


def count_chunks(reference_tokens: list[str], candidate_tokens: list[str]) -> int:
    """candidate에서 reference와 같은 순서로 이어지는 매칭 덩어리 수를 근사 계산한다."""
    ref_positions = {}
    for idx, token in enumerate(reference_tokens):
        ref_positions.setdefault(token, []).append(idx)

    matched_positions = []
    used_positions = set()
    for token in candidate_tokens:
        for pos in ref_positions.get(token, []):
            if pos not in used_positions:
                matched_positions.append(pos)
                used_positions.add(pos)
                break

    if not matched_positions:
        return 0

    chunks = 1
    for prev, cur in zip(matched_positions, matched_positions[1:]):
        if cur != prev + 1:
            chunks += 1
    return chunks


def calculate_meteor_ko(reference: str, candidate: str) -> dict:
    ref_tokens = tokenize_ko(reference)
    cand_tokens = tokenize_ko(candidate)

    matches = count_token_matches(ref_tokens, cand_tokens)
    if matches == 0:
        return {"meteor": 0.0, "precision": 0.0, "recall": 0.0, "penalty": 0.0}

    precision = matches / len(cand_tokens)
    recall = matches / len(ref_tokens)
    f_mean = (10 * precision * recall) / (recall + 9 * precision)

    chunks = count_chunks(ref_tokens, cand_tokens)
    penalty = 0.5 * (chunks / matches) ** 3
    meteor = f_mean * (1 - penalty)

    return {
        "meteor": meteor,
        "precision": precision,
        "recall": recall,
        "penalty": penalty,
        "matches": matches,
        "chunks": chunks,
    }


for name, generated in {
    "유사한 생성문": generated_good,
    "다른 생성문": generated_bad,
}.items():
    result = calculate_meteor_ko(reference, generated)
    print(f"[{name}] METEOR={result['meteor']:.3f}, "
          f"P={result['precision']:.3f}, R={result['recall']:.3f}, "
          f"Penalty={result['penalty']:.3f}")

[유사한 생성문] METEOR=0.605, P=0.625, R=0.625, Penalty=0.032
[다른 생성문] METEOR=0.130, P=0.400, R=0.250, Penalty=0.500


### 6.3 BERTScore 핵심 로직 구현

BERTScore는 정답과 생성문을 토큰 임베딩으로 변환한 뒤, 각 토큰이 상대 문장에서 가장 비슷한 토큰을 찾는 greedy matching을 수행합니다. 아래 예제는 `transformers`와 `torch`만으로 이 과정을 직접 보여줍니다. 모델은 한국어와 다국어 문장을 모두 처리하기 쉬운 `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`를 사용합니다.

In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()


def token_embeddings(text: str) -> torch.Tensor:
    encoded = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**encoded)

    embeddings = outputs.last_hidden_state.squeeze(0)
    attention_mask = encoded["attention_mask"].squeeze(0).bool()

    # [CLS], [SEP], padding 같은 특수 토큰은 비교에서 제외한다.
    special_mask = torch.tensor(
        tokenizer.get_special_tokens_mask(encoded["input_ids"].squeeze(0).tolist(), already_has_special_tokens=True),
        dtype=torch.bool,
    )
    keep_mask = attention_mask & ~special_mask
    return F.normalize(embeddings[keep_mask], p=2, dim=1)


def calculate_bertscore_like(reference: str, candidate: str) -> dict:
    ref_emb = token_embeddings(reference)
    cand_emb = token_embeddings(candidate)

    similarity = ref_emb @ cand_emb.T
    recall = similarity.max(dim=1).values.mean().item()
    precision = similarity.max(dim=0).values.mean().item()
    f1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


for name, generated in {
    "유사한 생성문": generated_good,
    "다른 생성문": generated_bad,
}.items():
    result = calculate_bertscore_like(reference, generated)
    print(f"[{name}] BERTScore-like F1={result['f1']:.3f}, "
          f"P={result['precision']:.3f}, R={result['recall']:.3f}")

e:\sw\dev\ai\modu_llm7\etf-bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\sw\dev\ai\modu_llm7\etf-bot\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\JSPark\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an

[유사한 생성문] BERTScore-like F1=0.940, P=0.937, R=0.942
[다른 생성문] BERTScore-like F1=0.326, P=0.347, R=0.307


### 6.4 결과를 표로 비교하기

In [4]:
import pandas as pd

rows = []
for name, generated in {
    "유사한 생성문": generated_good,
    "다른 생성문": generated_bad,
}.items():
    meteor = calculate_meteor_ko(reference, generated)
    bertscore = calculate_bertscore_like(reference, generated)
    rows.append({
        "case": name,
        "meteor": meteor["meteor"],
        "meteor_precision": meteor["precision"],
        "meteor_recall": meteor["recall"],
        "bertscore_f1": bertscore["f1"],
        "bertscore_precision": bertscore["precision"],
        "bertscore_recall": bertscore["recall"],
    })

pd.DataFrame(rows).round(3)

,case,meteor,meteor_precision,meteor_recall,bertscore_f1,bertscore_precision,bertscore_recall
0,유사한 생성문,0.605,0.625,0.625,0.940,0.937,0.942
1,다른 생성문,0.130,0.400,0.250,0.326,0.347,0.307


> 해석할 때는 두 지표의 관점을 분리해서 보아야 합니다. METEOR는 같은 토큰이 얼마나 겹치고 순서가 얼마나 유지되는지에 민감합니다. 반면 BERTScore는 표면 단어가 달라도 임베딩 공간에서 의미가 가까우면 높은 점수를 줄 수 있습니다. 따라서 한국어 LLM 답변 평가에서는 METEOR를 빠른 표면 검사용으로, BERTScore를 의미 유사도 검사용으로 함께 보는 방식이 실무적으로 유용합니다.